In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib, os, numpy as np, pandas as pd
from scipy.stats import rankdata, pearsonr

# Best feature set from Step 2 — update this after running Step 2
# Using top-50 as a reasonable default; change to your ablation winner
BEST_FEATURES_N = 50

def get_top_features(X, y_series, top_n, n_moons=60):
    all_feats = [c for c in X.columns if c.startswith("Feature_")]
    moons = X["moon"].values
    unique = np.unique(moons)
    if len(unique) > n_moons:
        unique = np.random.default_rng(42).choice(unique, n_moons, replace=False)
    ic_sum = np.zeros(len(all_feats))
    ic_cnt = np.zeros(len(all_feats))
    for m in unique:
        mask = moons == m
        yt = y_series.values[mask]
        if yt.std() < 1e-10: continue
        Xm = X.iloc[mask][all_feats].values
        for j in range(len(all_feats)):
            xf = Xm[:, j]
            if xf.std() < 1e-10: continue
            r = np.corrcoef(xf, yt)[0, 1]
            if not np.isnan(r):
                ic_sum[j] += r; ic_cnt[j] += 1
    safe = np.where(ic_cnt > 0, ic_cnt, 1)
    mean_ic = ic_sum / safe
    mean_ic[ic_cnt == 0] = 0.0
    order = np.argsort(np.abs(mean_ic))[::-1]
    return [all_feats[i] for i in order[:top_n]]

def evaluate_model(model_fn, label, feature_cols, X_tr, y_tr, X_te, y_te, splits):
    """model_fn(X_fit, y_fit) → fitted model with .predict(X)"""
    import time

    # Prepare training data
    df = X_tr.merge(y_tr[["moon","id","target"]], on=["moon","id"]).dropna(subset=["target"])
    df[feature_cols] = df.groupby("moon")[feature_cols].transform(
        lambda s: s.fillna(s.median()))
    df[feature_cols] = df[feature_cols].fillna(0.0)
    df["target_rank"] = df.groupby("moon")["target"].transform(
        lambda s: (s.rank(method="average") - 1) / max(len(s)-1, 1) - 0.5)

    t0 = time.perf_counter()
    model = model_fn(df[feature_cols], df["target_rank"])
    train_time = time.perf_counter() - t0

    # Infer per moon
    corrs = []
    for moon in splits["reduced_cloud"]:
        X_m = X_te[X_te["moon"] == moon].copy()
        for c in feature_cols:
            med = X_m[c].median()
            X_m[c] = X_m[c].fillna(med if not np.isnan(med) else 0.0)
        raw = model.predict(X_m[feature_cols])
        n = len(raw)
        final = (rankdata(raw) - 1) / max(n-1, 1) - 0.5
        y_m = y_te[y_te["moon"] == moon]
        merged = pd.DataFrame({"moon": X_m["moon"].values, "id": X_m["id"].values,
                                "prediction": final}).merge(
            y_m[["moon","id","target"]], on=["moon","id"])
        if len(merged) > 1:
            r, _ = pearsonr(merged["prediction"], merged["target"])
            corrs.append(r)

    mean_ic = float(np.mean(corrs)) if corrs else 0.0
    print(f"  [{label:30s}]  Mean IC = {mean_ic:+.4f}  "
          f"train={train_time:.0f}s  "
          f"per-moon: {[round(c,3) for c in corrs]}")
    return mean_ic


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
import xgboost as xgb
import time

print("Getting top features ...")
df_merge = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"]).dropna(subset=["target"])
feature_cols = get_top_features(df_merge, df_merge["target"], top_n=BEST_FEATURES_N)
print(f"Using top-{len(feature_cols)} features")

# Define model factories — each returns a fitted model
models = {
    "LGBM_baseline": lambda X, y: LGBMRegressor(
        n_estimators=200, learning_rate=0.05,
        random_state=42, n_jobs=-1, verbose=-1).fit(X, y),

    "LGBM_tuned": lambda X, y: LGBMRegressor(
        n_estimators=500, learning_rate=0.02, num_leaves=63,
        min_child_samples=50, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.05, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbose=-1).fit(X, y),

    "Ridge_quantile": lambda X, y: (
        lambda qt, ridge: type("M", (), {
            "predict": lambda self, Xp: ridge.predict(qt.transform(Xp))
        })()
    )(
        QuantileTransformer(output_distribution="normal", random_state=42).fit(X),
        Ridge(alpha=10.0).fit(
            QuantileTransformer(output_distribution="normal", random_state=42).fit_transform(X), y)
    ),

    "XGBoost": lambda X, y: xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.04, max_depth=6,
        subsample=0.7, colsample_bytree=0.7,
        random_state=42, n_jobs=-1, verbosity=0).fit(X, y),

    "HistGBM": lambda X, y: HistGradientBoostingRegressor(
        max_iter=300, max_leaf_nodes=63, max_bins=128,
        l2_regularization=0.1, learning_rate=0.04,
        early_stopping=False, random_state=42).fit(X, y),
}

print("\nRunning model comparison ...")
print("-" * 70)
results_models = {}
for label, model_fn in models.items():
    ic = evaluate_model(model_fn, label, feature_cols,
                        X_train, y_train, X_test_cloud, y_test_cloud, splits)
    results_models[label] = ic

print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
for label, ic in sorted(results_models.items(), key=lambda x: -x[1]):
    bar = "█" * int(max(ic, 0) * 200)
    print(f"  {label:30s}: {ic:+.4f}  {bar}")
